# AutoML for IoT anomaly detection: MEMS + N-BaIoT device 9

Runs the study in the background (detached processes), so closing the browser or restarting the kernel does **not** stop it.

**Order:** 1 Setup -> 2 Preview -> 3 Seed 0 (pilot) -> 4 Monitor -> 5 Verify seed 0 -> 6 Seeds 1-4 -> 7 Monitor -> 8 Aggregate.

**Kernel:** pick the `iot_automl` environment. If it is missing: `conda activate iot_automl && conda install ipykernel -y && python -m ipykernel install --user --name iot_automl --display-name "Python (iot_automl)"`.

**Rules:** use the same `THREADS` and `BUDGET` for every seed; never run seed 0 and seeds 1-4 at the same time; run each launch cell once (the guard blocks a second copy anyway).

## 1. Setup (edit the paths, then run cells 1.1 to 1.3 after every kernel restart)

In [ ]:
import os, sys, json, time, signal, subprocess
from pathlib import Path
import psutil
import pandas as pd

# ---------------- EDIT THESE ----------------
PROJECT_DIR = Path(os.environ.get("IOT_PROJECT", Path.cwd().parent / "pipeline" if (Path.cwd().parent / "pipeline").exists() else Path.home() / "iot_automl"))                 # folder with run_grid.sh
DATA_DIR    = Path(os.environ.get("IOT_DATA_DIR", Path.home() / "XAI_for_IoT_Systems" / "Datasets"))  # has mems_dataset.csv
OUT         = os.environ.get("IOT_OUT", "results")   # results folder (inside PROJECT_DIR)
THREADS     = 8        # SAME for every run (machine: 10 physical cores)
BUDGET      = 1800     # seconds per AutoML run, SAME for both frameworks
DEVICES     = "9"      # N-BaIoT device(s), space separated
# --------------------------------------------

PER_SEED = 19 + 12 * len(DEVICES.split())   # result files per seed (31 for one device)
assert (PROJECT_DIR / "run_grid.sh").exists(), f"PROJECT_DIR is wrong: {PROJECT_DIR}"
assert (DATA_DIR / "mems_dataset.csv").exists(), f"DATA_DIR is wrong (mems_dataset.csv not found): {DATA_DIR}"
# make the kernel's own python and java (conda env) the ones used by the background jobs
os.environ["PATH"] = str(Path(sys.executable).parent) + os.pathsep + os.environ["PATH"]
print("project:", PROJECT_DIR, "\ndata   :", DATA_DIR, "\nexpected result files per seed:", PER_SEED)

In [ ]:
# 1.2 Environment check
print("python:", sys.version.split()[0], "|", sys.executable)
for m in ["autosklearn", "h2o", "sklearn", "numpy", "pandas"]:
    try: print(f"{m:12s}", __import__(m).__version__)
    except Exception as e: print(f"{m:12s} MISSING ({e})")
try: print("java        ", subprocess.run(["java", "-version"], capture_output=True, text=True).stderr.splitlines()[0])
except FileNotFoundError: print("java         NOT FOUND -> conda install -c conda-forge openjdk=17")
print("CPUs usable:", len(os.sched_getaffinity(0)), "| physical:", psutil.cpu_count(logical=False),
      "| RAM GB:", round(psutil.virtual_memory().total / 1024**3))
assert sys.version_info[:2] == (3, 9), "Use the Python 3.9 environment (Auto-sklearn 0.15 needs it)"
assert THREADS <= psutil.cpu_count(logical=False), "THREADS exceeds physical cores"

In [ ]:
# 1.3 Helper functions
def _is_study_cmd(cmd):
    """True only for 'bash run_grid.sh' or 'python run_automl.py / run_baselines.py' (not editors, tail, tmux...)."""
    if not cmd or len(cmd) < 2: return False
    exe, script = os.path.basename(cmd[0]), os.path.basename(cmd[1])
    return (exe in ("bash", "sh") and script == "run_grid.sh") or            (exe.startswith("python") and script in ("run_automl.py", "run_baselines.py"))

def grid_processes():
    """Processes of this study owned by you."""
    me, out = psutil.Process().username(), []
    for p in psutil.process_iter(["pid", "cmdline", "username"]):
        if p.info["username"] == me and _is_study_cmd(p.info["cmdline"]):
            out.append((p.info["pid"], " ".join(p.info["cmdline"])[:120]))
    return out

def launch(seeds, log_name, extra_env=None):
    if grid_processes():
        raise RuntimeError("This study is already running (see grid_processes()). Not starting a second copy.")
    env = os.environ.copy()
    env.update(THREADS=str(THREADS), BUDGET=str(BUDGET), SEEDS=" ".join(map(str, seeds)),
               DEVICES=str(DEVICES), OUT=OUT, IOT_DATA_DIR=str(DATA_DIR), PYTHONUNBUFFERED="1")
    env.update(extra_env or {})
    log = open(PROJECT_DIR / log_name, "ab")
    p = subprocess.Popen(["bash", "run_grid.sh"], cwd=PROJECT_DIR, env=env, stdout=log,
                         stderr=subprocess.STDOUT, start_new_session=True)   # detached from the kernel
    print(f"started run_grid.sh (PID {p.pid}), seeds {seeds}, log: {PROJECT_DIR / log_name}")

def seed_counts():
    counts = {}
    for f in (PROJECT_DIR / OUT).glob("*.json"):
        s = int(f.stem.rsplit("__s", 1)[1]); counts[s] = counts.get(s, 0) + 1
    return dict(sorted(counts.items()))

def status(logs=("pilot.log", "full.log"), tail=6):
    res = PROJECT_DIR / OUT
    print(time.strftime("%a %d %b %H:%M:%S"))
    procs = grid_processes()
    print("running processes:", len(procs)); [print("  ", pid, c) for pid, c in procs[:4]]
    print("result files per seed (expected %d each):" % PER_SEED, seed_counts(), "| total", len(list(res.glob("*.json"))))
    print("runs in progress (locks):", [p.name.replace(".json.lock", "") for p in res.glob("*.lock")])
    newest = sorted(res.glob("*.json"), key=lambda p: p.stat().st_mtime, reverse=True)[:3]
    print("newest results:", [(p.name, time.strftime("%H:%M", time.localtime(p.stat().st_mtime))) for p in newest])
    for name in logs:
        lp = PROJECT_DIR / name
        if lp.exists():
            lines = lp.read_text(errors="ignore").splitlines()
            bad = [l for l in lines if "FAILED" in l or "Traceback" in l]
            print(f"--- {name}: {len(lines)} lines, FAILED/Traceback lines: {len(bad)}")
            print("\n".join(l[:140] for l in lines[-tail:]))

def watch(every=60, minutes=30):
    """Auto-refresh status; interrupt the cell (stop button) any time. The study keeps running."""
    from IPython.display import clear_output
    for _ in range(int(minutes * 60 / every)):
        clear_output(wait=True); status(); time.sleep(every)

def load_results(seeds=None):
    rows = []
    for f in (PROJECT_DIR / OUT).glob("*.json"):
        r = json.load(open(f))
        if seeds is not None and r["seed"] not in seeds: continue
        res, cfg = r["resources"], r.get("config") or {}
        rows.append(dict(tag=r["tag"], framework=r["framework"], seed=r["seed"],
                         macro_f1=round(r["metrics"]["macro_f1"], 4), accuracy=round(r["metrics"]["accuracy"], 4),
                         fit_s=res.get("wall_clock_fit_s"), peak_rss_mb=res.get("peak_rss_mb"),
                         models=res.get("n_models_evaluated"), threads=cfg.get("threads"),
                         ask_mem_mb=cfg.get("ask_mem_mb"), h2o_mem=cfg.get("h2o_mem")))
    return pd.DataFrame(rows).sort_values(["tag", "seed"]).reset_index(drop=True)

def stop_all(confirm=False):
    """Stop everything of this study (parents first, then their children) and remove stale locks.
    Call stop_all(confirm=True) to act; stop_all() only lists what it would stop."""
    procs = grid_processes(); print("would stop:", procs)
    if not confirm: return
    victims = []
    for pid, _ in procs:
        try:
            p = psutil.Process(pid); victims += [p] + p.children(recursive=True)
        except psutil.NoSuchProcess: pass
    for p in victims:
        try: p.terminate()
        except psutil.NoSuchProcess: pass
    _, alive = psutil.wait_procs(victims, timeout=10)
    for p in alive:
        try: p.kill()
        except psutil.NoSuchProcess: pass
    for lk in (PROJECT_DIR / OUT).glob("*.lock"): lk.unlink()
    print("stopped; remaining:", grid_processes())

## 2. Preview (no runs started)
Prints the commands and the run count. Expect `AutoML runs: 11` per seed.

In [ ]:
launch([0], "preview.log", extra_env={"DRY": "1"})
time.sleep(5)
print((PROJECT_DIR / "preview.log").read_text()[-1500:])

## 3. Seed 0 (pilot): about 5.7 h
Run this cell **once**. Skip it if seed 0 already has all result files.

In [ ]:
launch([0], "pilot.log")

## 4. Monitor (re-run any time, or use `watch()`)
The first 12 files are quick baselines; each AutoML run then takes ~31 min before its file appears.

In [ ]:
status()

In [ ]:
# optional auto-refresh (stop with the interrupt button; the study keeps running)
# watch(every=60, minutes=30)

## 5. Verify seed 0 before starting seeds 1-4
Expect: 31 files, no FAILED lines, `threads` = THREADS everywhere, `fit_s` roughly 1800-1900 for AutoML runs, `peak_rss_mb` well below RAM.

In [ ]:
d0 = load_results(seeds=[0])
auto = d0[~d0.framework.str.startswith("baseline")]
display(auto)
print("files for seed 0:", seed_counts().get(0, 0), "/", PER_SEED)
print("threads used:", sorted(auto.threads.dropna().unique()), "| ask_mem_mb:", sorted(auto.ask_mem_mb.dropna().unique()),
      "| h2o_mem:", sorted(auto.h2o_mem.dropna().unique()))
print("fit_s range:", auto.fit_s.min(), "-", auto.fit_s.max(), "| max peak RSS MB:", auto.peak_rss_mb.max())
READY = seed_counts().get(0, 0) >= PER_SEED and not grid_processes() and set(auto.threads.dropna()) == {THREADS}
print("READY FOR SEEDS 1-4:", READY)

## 6. Seeds 1-4: about 22.7 h
Starts only if seed 0 is complete (set `FORCE = True` to override). Run once. Same `THREADS` and `BUDGET` as seed 0.

In [ ]:
FORCE = False
assert FORCE or READY, "Seed 0 is not complete/verified yet (run cell 5)."
launch([1, 2, 3, 4], "full.log")

## 7. Monitor
Same as cell 4. Full study complete = 5 x PER_SEED files.

In [ ]:
status()

## 8. Aggregate (after all seeds are done)

In [ ]:
n = sum(seed_counts().values()); print("result files:", n, "| expected:", 5 * PER_SEED, "| per seed:", seed_counts())
r = subprocess.run([sys.executable, "aggregate.py", "--results", OUT, "--out", "tables"],
                   cwd=PROJECT_DIR, capture_output=True, text=True)
print(r.stdout[-400:], r.stderr[-400:])
df = pd.read_csv(PROJECT_DIR / "tables" / "all_runs.csv")
summary = (df.groupby(["dataset", "split", "train_dist", "framework"])
             .agg(seeds=("seed", "nunique"), macro_f1=("macro_f1", "mean"), macro_f1_std=("macro_f1", "std"),
                  balanced_acc=("balanced_accuracy", "mean"), fit_s=("wall_clock_fit_s", "mean"),
                  peak_rss_mb=("peak_rss_mb", "mean")).round(3))
display(summary)
print("groups with fewer than 5 seeds:", int((summary.seeds < 5).sum()))

## If something goes wrong
* `status()` shows processes, per-seed counts, runs in progress and the log tail.
* `stop_all()` lists what would be stopped; `stop_all(confirm=True)` stops it and removes stale locks. Then re-run the launch cell: finished runs are skipped.
* Never start a second copy; `launch()` refuses when this study is already running.